In [ ]:

import yfinance as yf
import pandas as pd
import numpy as np


def load_data():
    symbols = ["NVDA", "AMD"]
    data = yf.download(symbols, start="2018-01-01", end="2026-03-21")
    return data["Close"].copy()


def calculate_beta(data):
    beta = np.polyfit(data["AMD"], data["NVDA"], 1)[0]
    return beta


def compute_indicators(data, beta, window=30):
    data["Spread"] = data["NVDA"] - beta * data["AMD"]
    data["Mean"] = data["Spread"].rolling(window).mean()
    data["Std"] = data["Spread"].rolling(window).std()
    data["Z"] = (data["Spread"] - data["Mean"]) / data["Std"]
    return data


def generate_signals(data):
    data["RawSignal"] = np.nan

    data.loc[data["Z"] < -2, "RawSignal"] = 1
    data.loc[data["Z"] > 2, "RawSignal"] = -1
    data.loc[(data["Z"] > -0.5) & (data["Z"] < 0.5), "RawSignal"] = 0

    data["Signal"] = data["RawSignal"].ffill().shift(1).fillna(0)

    return data

def backtest(data, initial_capital=10000):
    capital = initial_capital
    position = 0

    entry_nvda = 0
    entry_amd = 0

    shares_nvda = 0
    shares_amd = 0

    for i in range(len(data)):
        row = data.iloc[i]


        if row["Signal"] == 1 and position == 0:
            position = 1

            shares_nvda = (capital / 2) / row["NVDA"]
            shares_amd = (capital / 2) / row["AMD"]

            entry_nvda = row["NVDA"]
            entry_amd = row["AMD"]

        elif row["Signal"] == -1 and position == 0:
            position = -1

            shares_nvda = (capital / 2) / row["NVDA"]
            shares_amd = (capital / 2) / row["AMD"]

            entry_nvda = row["NVDA"]
            entry_amd = row["AMD"]


        elif row["Signal"] == 0 and position != 0:

            if position == 1:
                pnl = (
                    (row["NVDA"] - entry_nvda) * shares_nvda
                    - (row["AMD"] - entry_amd) * shares_amd
                )

            elif position == -1:
                pnl = (
                    -(row["NVDA"] - entry_nvda) * shares_nvda
                    + (row["AMD"] - entry_amd) * shares_amd
                )

            capital += pnl
            position = 0

    return capital


def run_strategy():
    data = load_data()
    beta = calculate_beta(data)
    data = compute_indicators(data, beta)
    data = generate_signals(data)

    final_capital = backtest(data)

    print("\n=== RESULTS ===")
    print(f"Final Capital: {final_capital:.2f}")
    print(f"Return: {(final_capital - 10000)/10000*100:.2f}%")
    print(f"Hedge Ratio (beta): {beta:.4f}")


if __name__ == "__main__":
    run_strategy()